# 04a · Diagnóstico de qualidade das bases
**Objetivo:** antes de calcular qualquer KPI, saber se os dados são confiáveis.
Para cada tabela: linhas, colunas, vazios e duplicados. Depois, testes de regra de negócio
(cadastro de SKU, recebimento, endereços). **Nada é corrigido aqui**: os problemas são
listados em `reports/phase4/data_quality_issues.csv` e tratados (ou não) nas análises seguintes,
sempre com aviso.

*EN summary: data profiling of all tables plus business-rule checks; issues are listed, not fixed.*

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from dcanalysis import q, save, log

NB = "04a"
pd.set_option("display.width", 140)

## 1. Perfil geral de cada tabela
Vazios (`NULL`) nem sempre são erro: por exemplo, `arrival_date` vazio em PO aberto é esperado.
Por isso a tabela abaixo é só o ponto de partida; a interpretação vem nas seções seguintes.

In [2]:
tabelas = ["dim_sku", "dim_supplier", "dim_customer", "dim_location", "dim_lot",
           "fact_order_line", "fact_receipt", "fact_movement", "fact_inventory_daily",
           "fact_occupancy_daily", "fact_purchase_order", "fact_cycle_count", "fact_return",
           "aux_promotion", "aux_recall"]
perfil = []
for t in tabelas:
    df = q(f"SELECT * FROM {t}")
    vazios = df.isna().sum()
    perfil.append({
        "tabela": t, "linhas": len(df), "colunas": df.shape[1],
        "linhas_duplicadas": int(df.duplicated().sum()),
        "colunas_com_vazio": ", ".join(f"{c} ({n:,})" for c, n in vazios[vazios > 0].items()) or "-",
    })
perfil = pd.DataFrame(perfil)
save(perfil, "data_profile")
perfil

,tabela,linhas,colunas,linhas_duplicadas,colunas_com_vazio
0,dim_sku,600,19,0,"category (5), unit_weight_kg (4), pick_face_lo..."
1,dim_supplier,17,5,0,-
2,dim_customer,3150,5,0,-
3,dim_location,2940,11,0,"aisle (1,126), bay (1,126), level (1,126), slo..."
4,dim_lot,71072,5,0,-
5,fact_order_line,1505052,13,0,-
6,fact_receipt,105819,24,0,"stored_ts (8,934), dock_to_stock_min (8,934), ..."
7,fact_movement,1597871,12,0,"from_location_id (96,885), to_location_id (1,4..."
8,fact_inventory_daily,706306,5,0,-
9,fact_occupancy_daily,2193,5,0,-


**Leitura:** nenhuma linha duplicada inteira. Os vazios relevantes estão em `dim_sku`
(categoria, peso), `fact_receipt` (campos de putaway vazios = paletes recusados; temperatura
vazia = carga seca, sem termógrafo) e `aux_promotion` (data de comunicação ao planejamento).

## 2. Cadastro de SKU (`dim_sku`)
Regras testadas: peso unitário presente e plausível (0 < peso ≤ 50 kg), categoria preenchida,
descrição única, preço ≥ custo.

In [3]:
sku = q("SELECT * FROM dim_sku")
problemas = []

def registrar(tabela, regra, ids, gravidade, impacto):
    """Guarda um problema de qualidade encontrado (com exemplos de IDs)."""
    problemas.append({"tabela": tabela, "regra": regra, "ocorrencias": len(ids),
                      "exemplos": ", ".join(map(str, list(ids)[:8])),
                      "gravidade": gravidade, "impacto": impacto})

registrar("dim_sku", "peso unitário vazio", sku.loc[sku.unit_weight_kg.isna(), "sku_id"],
          "média", "frete, carga de empilhadeira e cubagem")
registrar("dim_sku", "peso unitário = 0", sku.loc[sku.unit_weight_kg == 0, "sku_id"],
          "média", "frete e cubagem subestimados")
registrar("dim_sku", "peso unitário > 50 kg (provável erro de unidade g/kg)",
          sku.loc[sku.unit_weight_kg > 50, "sku_id"], "alta",
          "palete calculado com toneladas: risco de bloquear alocação/segurança")
registrar("dim_sku", "categoria vazia", sku.loc[sku.category.isna(), "sku_id"],
          "baixa", "relatórios por categoria incompletos")
dup = sku[sku.description.duplicated(keep=False)].sort_values("description")
registrar("dim_sku", "descrição duplicada (SKUs diferentes, mesmo texto)", dup.sku_id,
          "média", "risco de separar o item errado; cadastro em duplicidade")
registrar("dim_sku", "preço de venda < custo", sku.loc[sku.unit_price < sku.unit_cost, "sku_id"],
          "alta", "margem negativa")

# Mostra os SKUs com peso suspeito (outliers são destacados, não excluídos)
sku.loc[sku.unit_weight_kg.isna() | (sku.unit_weight_kg <= 0) | (sku.unit_weight_kg > 50),
        ["sku_id", "sku_code", "description", "zone", "unit_weight_kg"]]

,sku_id,sku_code,description,zone,unit_weight_kg
14,14,SKU000015,Mercearia Fazenda Real Menta Econômico 350g,dry,NaN
57,57,SKU000058,Bebidas Sol Nascente Mel Tradicional 1kg,dry,NaN
199,199,SKU000200,Mercearia Nativa Menta Família 1kg,dry,1082.0
238,238,SKU000239,Higiene e Limpeza Ouro Fino Baunilha Tradicion...,dry,911.0
240,240,SKU000241,Mercearia Primavera Arroz Tradicional 1L,dry,NaN
242,242,SKU000243,Higiene e Limpeza Vale Verde Café Sabor Casa 1L,dry,0.0
245,245,SKU000246,Mercearia Fazenda Real Ervas Light 500g,dry,NaN
377,377,SKU000378,Mercearia Ouro Fino Abacaxi Integral 500ml,dry,1353.0
556,556,SKU000557,Carnes e Aves Vale Verde Ervas Econômico 500ml,frozen,0.0
579,579,SKU000580,Carnes e Aves Terra Boa Maçã Original 500g,frozen,0.0


In [4]:
# Descrições duplicadas: mesmo texto em SKUs diferentes
dup[["sku_id", "sku_code", "description", "zone", "status", "registered_date"]]

,sku_id,sku_code,description,zone,status,registered_date
435,435,SKU000436,Bebidas Cantinho Carne Tradicional 2kg,dry,ACTIVE,2024-04-20
357,357,SKU000358,Bebidas Cantinho Carne Tradicional 2kg,dry,ACTIVE,2020-11-13
168,168,SKU000169,Biscoitos e Doces Campo Bom Mel Light 2L,dry,ACTIVE,2021-08-15
348,348,SKU000349,Biscoitos e Doces Campo Bom Mel Light 2L,dry,ACTIVE,2021-03-17
397,397,SKU000398,Biscoitos e Doces Ponte Velha Baunilha Light 500g,dry,ACTIVE,2021-12-27
279,279,SKU000280,Biscoitos e Doces Ponte Velha Baunilha Light 500g,dry,ACTIVE,2021-08-23
525,525,SKU000526,Frios e Embutidos Vale Verde Cebola Light 1kg,chilled,ACTIVE,2021-06-18
85,85,SKU000086,Frios e Embutidos Vale Verde Cebola Light 1kg,dry,ACTIVE,2021-08-22
221,221,SKU000222,Higiene e Limpeza Lar Doce Cereja Light 1kg,dry,ACTIVE,2023-09-27
369,369,SKU000370,Higiene e Limpeza Lar Doce Cereja Light 1kg,dry,ACTIVE,2020-12-31


## 3. Recebimento (`fact_receipt`)
Regras do CD (de `config/parameters.yaml`):
- vida útil restante mínima de **70%** no recebimento;
- exposição máxima na doca: **30 min** (resfriado) e **15 min** (congelado);
- temperatura do caminhão: resfriado 0–8 °C, congelado ≤ −18 °C.

In [5]:
rec = q("""SELECT receipt_id, supplier_id, zone, status, remaining_life_pct,
                  dock_exposure_min, truck_temp_c FROM fact_receipt""")
aceito = rec[rec.status == "ACCEPTED"]

abaixo = aceito[aceito.remaining_life_pct < 70]
registrar("fact_receipt", "palete ACEITO com vida útil < 70% (regra de recebimento furada)",
          abaixo.receipt_id, "alta", "perda por vencimento e ruptura de FEFO no cliente")

limite = aceito.zone.map({"chilled": 30, "frozen": 15})
exposto = aceito[aceito.dock_exposure_min > limite]
registrar("fact_receipt", "exposição na doca acima do limite (frio)", exposto.receipt_id,
          "alta", "quebra da cadeia do frio, risco sanitário")

temp_fora = aceito[((aceito.zone == "chilled") & ((aceito.truck_temp_c < 0) | (aceito.truck_temp_c > 8))) |
                   ((aceito.zone == "frozen") & (aceito.truck_temp_c > -18))]
registrar("fact_receipt", "palete aceito de caminhão com temperatura fora da faixa",
          temp_fora.receipt_id, "alta", "produto possivelmente comprometido entrou no estoque")

resumo_rec = (aceito.assign(abaixo_70=aceito.remaining_life_pct < 70,
                            exposicao_acima=aceito.dock_exposure_min > limite)
              .groupby("zone")[["abaixo_70", "exposicao_acima"]].mean().mul(100).round(1))
resumo_rec.columns = ["% aceitos com vida < 70%", "% acima da exposição máx."]
resumo_rec

,% aceitos com vida < 70%,% acima da exposição máx.
zone,,
chilled,3.2,26.0
dry,9.2,0.0
frozen,0.0,60.8


In [6]:
# Quem são os fornecedores dos paletes aceitos abaixo de 70%?
(abaixo.groupby("supplier_id").size().rename("paletes_abaixo_70").sort_values(ascending=False)
 .to_frame().join(aceito.groupby("supplier_id").size().rename("paletes_aceitos"))
 .assign(pct=lambda d: (100 * d.paletes_abaixo_70 / d.paletes_aceitos).round(1)).head(6))

,paletes_abaixo_70,paletes_aceitos,pct
supplier_id,,,
3,5268,5850,90.1
13,826,946,87.3
11,1,11286,0.0


## 4. Endereços, pedidos e promoções

In [7]:
loc = q("SELECT * FROM dim_location")
# Endereços sem rua/prédio: confere se são só transbordo (chão) e caixas de unidade
sem_coord = loc[loc.aisle.isna()].location_type.value_counts()
print("Endereços sem rua/prédio por tipo:\n", sem_coord)

# Pedidos fora do período (warm-up)
ol = q("SELECT MIN(order_date) ini, MAX(order_date) fim, SUM(order_date < '2024-11-01') linhas_antes FROM fact_order_line")
print(ol)
antes = q("SELECT order_line_id FROM fact_order_line WHERE order_date < '2024-11-01'")
registrar("fact_order_line", "linhas com pedido antes de 01/11/2024 (resto do warm-up)",
          antes.order_line_id, "baixa", "excluir dos KPIs mensais")

# Coerência categoria x zona: a categoria "manda" na temperatura (ex.: Frios só no resfriado).
# A zona mais comum de cada categoria é tomada como a correta; as exceções são sinalizadas.
zona_tipica = sku.groupby("category").zone.agg(lambda z: z.mode()[0])
incoerente = sku[sku.category.notna() & (sku.zone != sku.category.map(zona_tipica))]
registrar("dim_sku", "zona de temperatura incoerente com a categoria", incoerente.sku_id,
          "alta", "produto refrigerado guardado no seco (ou o contrário)")
print("SKUs com zona incoerente:\n", incoerente[["sku_id", "description", "category", "zone"]])

promo = q("SELECT * FROM aux_promotion")
nao_com = promo[promo.communicated_to_planning_date.isna()]
tarde = promo[promo.communicated_to_planning_date > promo.start_date]
registrar("aux_promotion", "promoção sem data de comunicação ao planejamento", nao_com.promo_id,
          "alta", "demanda de promoção não planejada -> ruptura")
registrar("aux_promotion", "promoção comunicada depois do início", tarde.promo_id,
          "média", "planejamento sem tempo de reagir")
print(f"Promoções: {len(promo)} | sem comunicação: {len(nao_com)} ({len(nao_com)/len(promo):.0%})")

Endereços sem rua/prédio por tipo:
 location_type
overflow    726
unit_bin    400
Name: count, dtype: int64


          ini         fim  linhas_antes
0  2024-10-31  2026-10-31           155
SKUs com zona incoerente:
 Empty DataFrame
Columns: [sku_id, description, category, zone]
Index: []
Promoções: 1152 | sem comunicação: 651 (57%)


## 5. Lista consolidada de problemas

In [8]:
issues = pd.DataFrame(problemas)
# Ordena por gravidade (alta > média > baixa) e depois por número de ocorrências
issues["gravidade"] = pd.Categorical(issues.gravidade, ["alta", "média", "baixa"], ordered=True)
issues = issues.sort_values(["gravidade", "ocorrencias"], ascending=[True, False])
issues = issues[issues.ocorrencias > 0]
save(issues, "data_quality_issues")
log(NB, f"perfil de {len(tabelas)} tabelas; {len(issues)} regras com ocorrência -> data_quality_issues.csv")
issues

,tabela,regra,ocorrencias,exemplos,gravidade,impacto
7,fact_receipt,exposição na doca acima do limite (frio),15073,"129, 130, 131, 136, 137, 138, 139, 254",alta,"quebra da cadeia do frio, risco sanitário"
6,fact_receipt,palete ACEITO com vida útil < 70% (regra de re...,6095,"19, 20, 21, 22, 23, 24, 30, 155",alta,perda por vencimento e ruptura de FEFO no cliente
8,fact_receipt,palete aceito de caminhão com temperatura fora...,2179,"123, 124, 295, 296, 297, 298, 299, 300",alta,produto possivelmente comprometido entrou no e...
11,aux_promotion,promoção sem data de comunicação ao planejamento,651,"52, 53, 54, 55, 56, 57, 58, 59",alta,demanda de promoção não planejada -> ruptura
2,dim_sku,peso unitário > 50 kg (provável erro de unidad...,4,"199, 238, 377, 599",alta,palete calculado com toneladas: risco de bloqu...
4,dim_sku,"descrição duplicada (SKUs diferentes, mesmo te...",18,"435, 357, 168, 348, 397, 279, 525, 85",média,risco de separar o item errado; cadastro em du...
0,dim_sku,peso unitário vazio,4,"14, 57, 240, 245",média,"frete, carga de empilhadeira e cubagem"
1,dim_sku,peso unitário = 0,3,"242, 556, 579",média,frete e cubagem subestimados
9,fact_order_line,linhas com pedido antes de 01/11/2024 (resto d...,155,"113673, 113674, 113675, 113676, 113677, 113678...",baixa,excluir dos KPIs mensais
3,dim_sku,categoria vazia,5,"211, 239, 311, 464, 478",baixa,relatórios por categoria incompletos


## Conclusões e tratamento adotado nas próximas análises
- **Cadastro:** pesos vazios, zerados e na casa das centenas de kg (erro de unidade) — não
  afetam os KPIs de unidades, mas afetam cubagem e frete. **Ação:** saneamento do cadastro (Fase 9).
  Nas análises, peso não é usado.
- **Descrições duplicadas:** SKUs diferentes com mesmo texto. Mantidos como SKUs distintos
  (códigos diferentes), mas sinalizados para revisão.
- **Recebimento:** existem paletes **aceitos** com vida útil < 70% e exposição na doca acima do
  limite no frio. Isso não é erro de dado, é **falha de processo** — vai para a investigação (04b).
- **Promoções sem comunicação ao planejamento:** falha de processo; impacto medido em 04b/04d.
- **Linhas antes de 01/11/2024:** poucas; excluídas dos KPIs mensais (filtro de data).